In [ ]:
#| echo: true

import os

from examples.example_2_branching_multistep.ex_2_model_classes import Trial, g

In [ ]:
#| echo: true

g.sim_duration = 3000
g.number_of_runs = 1

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "../examples/example_2_branching_multistep/ex_2_model_classes.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
#| echo: true

my_trial = Trial()

my_trial.run_trial()

In [ ]:
#| echo: true
my_trial.all_event_logs.head(10)

In [ ]:
#| echo: true

my_trial.all_event_logs.event_type.value_counts()

In [ ]:
#| echo: true

my_trial.all_event_logs.event.value_counts()

In [ ]:
#| echo: true

# First, identify all patients who have a 'depart' event
# patients_with_depart = my_trial.all_event_logs[my_trial.all_event_logs['event'].str.contains('depart')]['patient'].unique()

# Then filter the original DataFrame to only include those patients
# filtered_df = my_trial.all_event_logs[my_trial.all_event_logs['patient'].isin(patients_with_depart)]

logs_transformed = my_trial.all_event_logs[
    ~my_trial.all_event_logs["event"].str.contains("wait")
].copy()
# logs_transformed = filtered_df[~filtered_df['event'].str.contains('wait')].copy()
logs_transformed = logs_transformed[
    logs_transformed["event_type"].isin(["resource_use", "resource_use_end"])
].copy()
logs_transformed["event_stage"] = logs_transformed["event_type"].apply(
    lambda x: "complete" if "end" in x else "start"
)
logs_transformed["event_name"] = logs_transformed["event"].str.replace(
    "_begins|_complete", "", regex=True
)
logs_transformed["resource_id_full"] = logs_transformed.apply(
    lambda x: f"{x['event_name']}_{x['resource_id']:.0f}", axis=1
)
logs_transformed = logs_transformed.sort_values(["run", "time"], ascending=True)
# logs_transformed["activity_id"] = (
#     logs_transformed.groupby(["run", "patient", "event_name"]).ngroup() + 1
# )

# logs_transformed = logs_transformed.sort_values(["run", "patient", "activity_id", "event_stage"], ascending=[True, True, True, False])

# Sort the data by run, patient, time, and event_name to handle tied start times
logs_transformed = logs_transformed.sort_values(
    ["run", "patient", "time", "event_name"]
)

# Get the first occurrence of each activity (the start event)
first_occurrences = (
    logs_transformed[logs_transformed["event_stage"] == "start"]
    .drop_duplicates(["run", "patient", "event_name"])
    .copy()
)

# Sort by time within each run to determine the proper sequence
first_occurrences = first_occurrences.sort_values(["run", "time", "event_name"])

# Assign sequential activity_id within each run
first_occurrences["activity_id"] = first_occurrences.groupby("run").cumcount() + 1

# Merge the activity_id back to the main DataFrame
logs_transformed = logs_transformed.merge(
    first_occurrences[["run", "patient", "event_name", "activity_id"]],
    on=["run", "patient", "event_name"],
    how="left",
)

# Sort for final ordering
logs_transformed = logs_transformed.sort_values(
    ["run", "patient", "activity_id", "event_stage"],
    ascending=[True, True, True, False],
)
logs_transformed.head(50)

In [ ]:
#| echo: true

logs_transformed[
    (logs_transformed["run"] == 1) & (logs_transformed["activity_id"] == 26)
]

In [ ]:
#| echo: true

logs_transformed[logs_transformed["activity_id"] == 26].sort_values("run").head(30)

In [ ]:
#| echo: true
logs_transformed.sort_values("activity_id").head(20)

In [ ]:
#| echo: true
logs_transformed[["event_name", "event_stage", "event_type"]].value_counts()

In [ ]:
logs_transformed.event.value_counts()

For ease, now let's save these results as a file that we can load into R. 

We could use a csv for easy interoperability. Alternatively, we could use something like Feather or Parquet, which are usable by both R and Python while retaining data types.

For ease of use and long-term readbility, we will use csv in this case. 

In [ ]:
logs_transformed.to_csv("simulation_logs_for_bupar.csv", index=False)